# 텍스트 분석 전체 흐름 한 장 정리

## 1. 한눈에 보는 전체 흐름

텍스트 분석은 **수집 → 정제 → 분석 방법 선택 → 평가·개선**의 한 흐름이고, 분석 방법은 목적에 따라 TF-IDF·임베딩·LLM 중에서 고르거나 조합한다.

```
                 1. 데이터 수집
            API·크롤링 → raw.csv (원본 보존)
                       │
                 2. 정제·품질 검증
        중복·결측·공백 정리, 문서 ID → clean.csv
                       │
       ┌───────────────┼───────────────┐
  3-A TF-IDF      3-B 임베딩        3-C LLM    
 형태소·토큰화     청크 → 벡터       프롬프트 → API   
   → 키워드       → 코사인 Top-K     → JSON 출력
       │          │         │          │
       └────┬─────┘         └────┬─────┘
3-D BERTopic = 임베딩 + c-TF-IDF   3-E RAG = 임베딩 검색 + LLM
의미로 묶고 단어로 이름 붙이기         찾은 원문을 근거로 답변
            │                         │
            └────────────┬────────────┘
                   ④ 평가·개선
   정확성·일관성·처리 성공률 → 원인 수정 → 같은 데이터로 재평가
```

어느 갈래로 가든 마지막엔 평가로 모이고, 문제가 보이면 ②·③으로 돌아가 고친다.

## 2. TF-IDF · 임베딩 · LLM: 순서가 아니라 선택

세 방법은 차례로 다 거치는 단계가 아니라 **문제 정의에 맞게 고르는 도구**다. 필요하면 서로의 약점을 보완하려고 조합한다.

| 풀고 싶은 문제 | 우선 고르는 방법 | 이유 |
| --- | --- | --- |
| 문서에서 특징적인 단어는? | TF-IDF | 단어 중요도를 바로 계산, 빠르고 해석 쉬움 |
| 이 질문과 비슷한 문서는? | 임베딩 | 단어가 달라도 의미로 비교 |
| 분류·요약·추출을 해줘 | LLM | 문맥을 이해하고 결과를 생성 |
| 전체에 어떤 주제들이 있나? | BERTopic | 임베딩으로 묶고 c-TF-IDF로 이름 붙이기 |
| 사내 문서를 근거로 답해줘 | RAG | 임베딩으로 찾고 LLM이 답변 |

**자주 쓰는 조합**

| 조합 | 어떻게 | 왜 |
| --- | --- | --- |
| BERTopic | 임베딩 + c-TF-IDF | 의미로 묶고 묶은 군집에 단어로 이름 붙이기 |
| RAG | 임베딩 검색 + LLM | 근거 찾기 + 답 쓰기 |
| 하이브리드 검색 | 임베딩 + BM25(TF-IDF 계열) | 날짜·코드처럼 정확한 단어도 잡기 |
| LLM 라벨 → 작은 모델 | LLM으로 라벨 생성 → 임베딩 + 분류기 학습 | 대량 분류를 싸고 빠르게 |
| 비교 기준 | TF-IDF + 로지스틱 회귀 vs LLM | LLM이 정말 더 나은지 확인 |

**판단 순서**

1. 무엇을 알고 싶은가? (키워드 / 비슷한 것 / 분류·요약 / 전체 주제 / 근거 있는 답)
2. 그 목적에 가장 단순한 방법은?
3. 부족한 부분만 다른 방법으로 보완 (조합)
4. 같은 평가 데이터로 비교해서 결정

## 3. 배운 개념이 흐름 어디에 놓이나

| 개념 | 위치 | 하는 일 |
| --- | --- | --- |
| 크롤링, API, has_next, Next 링크 | ① 수집 | 원본 텍스트 확보 |
| 중복·결측·공백 정리, 스키마, 문서 ID, 메타데이터 | ② 정제 | 분석 가능한 clean.csv |
| 형태소 분석, 토큰화(내가 직접) | ③-A 앞 | 단어를 일관된 단위로 세기 |
| TF-IDF 행렬, 키워드 | ③-A | 문서별 특징 단어 |
| 청크 분할 | ③-B 앞 (RAG 준비) | 긴 문서를 검색 단위로 자르기 |
| 임베딩 벡터, 코사인 유사도, Top-K | ③-B | 의미가 비슷한 문서 찾기 |
| 프롬프트(RICO, Zero/Few-shot), 시스템·유저 메시지 | ③-C | 무엇을 어떤 기준으로 시킬지 |
| LLM API, Structured Outputs, 처리 상태 | ③-C | 프로그램에서 호출하고 형식 고정 |
| 워크플로우(단일·순차·분기) | ③-C 안 | LLM 호출을 몇 번, 어떤 순서로 |
| c-TF-IDF | BERTopic | 임베딩으로 묶은 그룹에 단어로 이름 붙이기 |
| 컨텍스트, 리랭커, 하이브리드 검색 | RAG | 찾은 원문을 LLM 입력에 넣기 |
| 정확성·일관성·처리 성공률, 평가 데이터 | ④ 평가·개선 | 맞는지 재고 고치기 |

## 4. 문서 · 청크 · 토큰, 그리고 토큰화가 일어나는 세 곳

크기는 **문서 > 청크 > 토큰** 순이고, 청크는 내가 자르고 토큰은 상황에 따라 내가 또는 모델이 자른다.

| 단위 | 크기 | 누가 자르나 | 어디서 |
| --- | --- | --- | --- |
| 문서 | PDF 한 개, 명언 한 줄 | 원래 단위 | ① 수집 |
| 청크 | 문단 하나 | 내가 규칙으로 (크기·겹침) | RAG 준비 |
| 토큰 | 단어 / 단어 조각 / 문장부호 | 아래 표 | 여러 곳 |

**토큰화는 세 군데서 일어나고, 성격이 다르다**

| 위치 | 누가 | 목적 | 내가 조절? |
| --- | --- | --- | --- |
| TF-IDF 앞 | 내가 형태소 분석기·공백 분리로 | 단어를 세기 위해 (상품이 → 상품) | O |
| 임베딩 모델 안 | 모델이 자동으로 (서브워드) | 토큰 → 어텐션 → 풀링 → 벡터 | X |
| LLM 안 | 모델이 자동으로 (서브워드) | 다음 토큰 예측 | X (대신 토큰 수 = 비용) |

LLM에 넣기 전에 형태소 분석을 할 필요는 없다. 모델이 자기 방식으로 토큰화하고, 오히려 원문 맥락을 그대로 주는 게 좋다.

**청크로 자르는 이유**

- 긴 문서를 통째로 임베딩하면 평균 풀링으로 내용이 뭉개진다
- LLM 컨텍스트 윈도우에 문서 전체를 넣기엔 길고 비싸다
- 너무 작으면 맥락이 잘리고, 너무 크면 주제가 섞인다 → 문단 단위 + 조금씩 겹치기

## 5. RAG 경로 확대: 청크·토큰화·컨텍스트가 쓰이는 곳

RAG는 **미리 준비**(청크 → 임베딩 → 저장)와 **질문마다**(검색 → 원문 조회 → 프롬프트 → LLM) 두 단계로 나뉘고, LLM에는 벡터가 아니라 **원문**이 들어간다.

```
[미리 준비 · 문서마다 한 번]
원본 문서 ──→ 청크 분할 ──→ 임베딩 모델 ──→ 벡터 DB
PDF·CSV·FAQ   문단 단위+겹침   내부 토큰화→벡터   벡터·원문·ID·메타
                                                     │
[질문이 올 때마다]                                     ↓
질문 ──→ 질문 임베딩 ──→ Top-K 검색 ──→ 원문 조회
사용자 입력  문서와 같은 모델  코사인 + BM25   ID로 원문·출처
                                              │
검증·저장 ←── 구조화 출력 ←── LLM ←────── 프롬프트 구성
status·ID merge  JSON 스키마   토큰화→다음 토큰  지시 + 문맥 + 질문
   │
   ↓
평가·개선은 두 겹: 맞는 원문을 찾았나 + 그 원문으로 맞게 답했나 → 수정 후 재평가
```

- 프롬프트 구성 칸이 컨텍스트를 채우는 곳이고, RAG의 A(Augmented)가 바로 이 칸이다.
- 질문 임베딩은 문서와 같은 모델이어야 한다. 모델을 바꾸면 문서 전체를 재임베딩한다.
- 이 그림은 검색 후 LLM 1회 호출인 단일 호출 RAG다. LLM 앞에 질문 다듬기 호출을 넣으면 순차 처리, 문서 길과 SQL 길로 나누면 조건부 분기, 그 길을 LLM이 고르면 에이전트다.

## 6. 세 가지 흐름의 관계

텍스트 분석 흐름이 가장 큰 상자이고, 그 안에서 LLM을 고르면 LLM 분석 흐름이, 그 안의 API 요청 칸을 확대한 것이 워크플로우다.

```
┌─ ① 텍스트 분석 전체 흐름 (과목 전체) ─────────────────────────────┐
│ 수집 → 정제 → 방법 선택 (TF-IDF / 임베딩 / LLM) → 평가·개선        │
│  ┌─ ② LLM 분석 흐름 (3~4장) · 방법으로 LLM을 골랐을 때 ─────────┐  │
│  │ 목적·기준 → 프롬프트 → API 요청 → 구조화 출력 → 여러 문서 → 검증·저장 │  │
│  │  ┌─ ③ 워크플로우 (5장 1강) · ②의 API 요청 칸을 확대 ──────┐  │  │
│  │  │ API 요청을 몇 번, 어떤 순서·조건으로 할까?               │  │  │
│  │  │ 단일 호출 · 순차 처리 · 조건부 분기 (LLM이 고르면 에이전트) │  │  │
│  │  └──────────────────────────────────────────────────┘  │  │
│  └──────────────────────────────────────────────────────┘  │
│  ┌─ ④ 평가·개선 (5장 2강) · ①의 마지막 칸을 확대 ───────────┐  │
│  │ 결과가 맞는지 재고, 원인이 있는 층으로 돌아가 고친다          │  │
│  └──────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────┘
```

| 지금 고민이 이거라면 | 해당 층 |
| --- | --- |
| TF-IDF로 할까, 임베딩으로 할까, LLM으로 할까? | ① 전체 흐름 |
| 프롬프트를 어떻게 쓰고, 결과를 어떤 형식으로 받을까? | ② LLM 분석 흐름 |
| LLM을 한 번 부를까, 두 번 부를까, 결과 보고 갈라질까? | ③ 워크플로우 |
| 이 결과가 맞나? 뭘 고쳐야 하나? | ④ 평가·개선 |

## 7. 딥러닝과의 연결, 그리고 내 역할

텍스트 분석에서는 LLM을 만드는 게 아니라, **이미 학습된 모델을 API로 불러 고르고 연결하고 검증한다.** API는 임시 대용이 아니라 실무에서도 쓰는 정상적인 방법이다.

| 과목 | 관점 | 하는 일 |
| --- | --- | --- |
| 딥러닝 1~6장 | 모델이 어떻게 만들어지는지 이해 | CNN·Transformer 직접 학습, 역전파 |
| 텍스트 분석 3~5장 | 만들어진 LLM 활용 | 프롬프트, API, 구조화 출력, 워크플로우, 평가 |
| 이후 RAG·Agent | LLM을 부품으로 시스템 구축 | 검색 연결, 도구 사용, 에이전트 |

**그림의 칸 ↔ 딥러닝에서 배운 것**

| 칸 | 연결 |
| --- | --- |
| 임베딩 모델 내부 | 토큰 임베딩 → Transformer Encoder → 평균 풀링 |
| LLM 내부 | Decoder-only, Masked 어텐션, 다음 토큰 Softmax |
| 두 모델 모두 | 사전학습 후 고정해서 사용 (6장 Feature Extraction) |
| 평가 | Train/Val/Test, Macro F1, Few-shot 예시와 평가 데이터 분리 |

**역전파는 어디서?**

- RAG·워크플로우 사용 중에는 없다: 모델은 고정, 순전파(추론)만
- 각 모델(임베딩·리랭커·LLM)은 과거에 따로, 자기 손실로 학습됐다
- 내가 역전파를 쓰는 곳: 임베딩 위에 붙이는 작은 분류기 학습, 특정 부품 Fine-tuning
- 개선은 대부분 설정·프롬프트를 바꾸고 같은 평가 데이터로 재는 방식

**내 역할**: 목적에 맞는 모델을 고르고, 데이터를 준비하고, 검색·프롬프트·DB로 연결하고, 결과를 평가해 개선한다.

## 8. 핵심 요약

**핵심 요약**

1. 과목 전체는 수집 → 정제 → (TF-IDF / 임베딩 / LLM 중 선택·조합) → 평가·개선의 한 흐름이다.
2. 세 방법은 차례로 거치는 단계가 아니라 문제에 맞게 고르는 도구이고, BERTopic·RAG·하이브리드는 조합이다.
3. 토큰화는 TF-IDF 앞(내가 직접)과 임베딩·LLM 안(모델이 자동)에서, 청크 분할은 RAG 준비에서, 컨텍스트는 프롬프트 구성에서 쓰인다.
4. RAG는 LLM을 바꾸지 않고, 밖에서 찾은 원문을 입력에 넣는 방식이다. 검색은 벡터로, LLM에는 문서 ID로 찾은 원문을 넣는다.
5. 워크플로우는 LLM 분석 흐름 안의 "API를 어떻게 부를까" 칸이고, Python이 순서와 조건을 제어한다. LLM이 길을 고르면 에이전트다.
6. 평가는 처리·구조 검증과 내용 평가를 나눠서, 같은 평가 데이터로 수정 전후를 비교한다.